# Module 3 — Classification (Telco churn): commented walkthrough

Alexey's lecture code (lessons 3.2 – 3.12) with comments that say **what each step is for and why**.
The original, uncommented notebook is untouched in `../notebooks/`.

Differences from the lecture:
- the data is read from the local file instead of `wget`;
- text columns are found with `select_dtypes` (the lecture's `== 'object'` finds nothing on pandas 3, where text columns have dtype `str`);
- the hand-typed numbers in 3.11 are computed in code, so they match *your* model;
- the lecture uses `random_state=1`; **HW3 pins `random_state=42`** and different solver settings, so HW3 numbers will differ from these.

## 3.2 Data preparation

In [ ]:
import pandas as pd                 # tables (DataFrame): loading, filtering, grouping
import numpy as np                  # array maths: used later for the sigmoid
import matplotlib.pyplot as plt     # plotting: used for the sigmoid curve

%matplotlib inline

In [ ]:
# The lecture downloads this file with wget; it is already in this folder.
# One row = one telecom customer. The column 'Churn' says whether they left.
df = pd.read_csv('data-week-3.csv')
df.head()

In [ ]:
# GOAL: make names and values consistent (lowercase, no spaces) so that later code can write
# df.multiplelines or compare with 'no_phone_service' without caring about capitalisation or spaces.

# 1) Column names: 'MultipleLines' -> 'multiplelines'
df.columns = df.columns.str.lower().str.replace(' ', '_')

# 2) Find the text columns. select_dtypes works on old pandas (dtype 'object') AND pandas 3 (dtype 'str').
#    Using  df.dtypes == 'object'  on pandas 3 selects nothing, so the loop below would silently do nothing.
categorical_columns = list(df.select_dtypes(include=['object', 'string']).columns)

# 3) Clean the VALUES of those columns: 'No phone service' -> 'no_phone_service'
for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_')

df.multiplelines.unique()   # sanity check: expect 'no_phone_service', 'no', 'yes'

In [ ]:
df.dtypes        # 'seniorcitizen' is already 0/1; 'totalcharges' is TEXT although it should be a number

In [ ]:
df.isna().sum()  # reports no missing values, yet totalcharges is text: the 'missing' values are hidden as blank strings

In [ ]:
# Turn totalcharges into numbers. errors='coerce' converts anything that can't be parsed into NaN
# instead of raising an error (the blanks were cleaned to '_' above, which is still not a number).
tc = pd.to_numeric(df.totalcharges, errors='coerce')

# Who are the unparseable rows? Expect brand-new customers: tenure 0, so nothing has been billed yet.
df[tc.isna()][['customerid', 'tenure', 'monthlycharges', 'totalcharges']]

In [ ]:
df.totalcharges = tc                          # keep the numeric version
df.totalcharges = df.totalcharges.fillna(0)   # 0 is sensible: those customers have tenure 0, i.e. no charges so far

In [ ]:
# The target must be numeric: 'yes' -> 1 (churned), 'no' -> 0 (stayed).
# The comparison gives True/False; astype(int) turns that into 1/0.
df.churn = (df.churn == 'yes').astype(int)
df.churn.head()

## 3.3 Setting up the validation framework

In [ ]:
from sklearn.model_selection import train_test_split

# Two splits give 60% train / 20% validation / 20% test:
#   1) take 20% off as the test set (locked away until the very end),
#   2) split the remaining 80% again, 25% of it (= 20% of the total) becomes validation.
# random_state fixes the shuffle so the result is reproducible. (HW3 uses 42; the lecture uses 1.)
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

len(df_train), len(df_val), len(df_test)

In [ ]:
# The shuffle leaves the original row labels in scrambled order. Reset them to 0..n-1;
# drop=True throws the old labels away instead of keeping them as an extra column.
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [ ]:
# Separate the target (y) from the features. Deleting the column from the dataframes matters:
# if 'churn' stayed among the features the model would just read the answer (target leakage).
y_train = df_train.churn.values
y_val = df_val.churn.values
y_test = df_test.churn.values

del df_train['churn']
del df_val['churn']
del df_test['churn']

## 3.4 EDA

In [ ]:
# From now on we explore df_full_train (train + validation) and never touch df_test.
df_full_train = df_full_train.reset_index(drop=True)
df_full_train.isnull().sum()    # missing values: none left after the totalcharges fix

In [ ]:
# Churn is 0/1, so its MEAN is the share of customers who churned: the base rate.
# It tells us the classes are imbalanced (more stayers than leavers): a model that always says "stays"
# would already score well on accuracy, so accuracy alone can mislead.
print(df_full_train.churn.value_counts(normalize=True))
global_churn = df_full_train.churn.mean()
global_churn

In [ ]:
# Split the features by type, because they are analysed differently below.
numerical = ['tenure', 'monthlycharges', 'totalcharges']

categorical = [
    'gender', 'seniorcitizen', 'partner', 'dependents', 'phoneservice', 'multiplelines',
    'internetservice', 'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport',
    'streamingtv', 'streamingmovies', 'contract', 'paperlessbilling', 'paymentmethod',
]

df_full_train[categorical].nunique()   # how many distinct values each categorical feature has (all small)

## 3.5 Feature importance: churn rate and risk ratio

In [ ]:
# Churn rate inside one group = mean of churn over just that group's rows.
churn_female = df_full_train[df_full_train.gender == 'female'].churn.mean()
churn_male = df_full_train[df_full_train.gender == 'male'].churn.mean()

# Difference from the overall rate: close to 0 means the group behaves like everyone else.
# (The lecture computes global - group here; the loop below uses group - global. Mind the sign.)
print('female', global_churn - churn_female, ' male', global_churn - churn_male)

In [ ]:
churn_partner = df_full_train[df_full_train.partner == 'yes'].churn.mean()
churn_no_partner = df_full_train[df_full_train.partner == 'no'].churn.mean()

# Risk ratio = group churn rate / overall churn rate.
#   > 1  the group churns MORE than average     < 1  it churns LESS     ~ 1  no information
print('risk partner', churn_partner / global_churn, ' risk no partner', churn_no_partner / global_churn)

In [ ]:
from IPython.display import display

# Same calculation for every categorical feature and every value of it (SQL: GROUP BY feature).
# 'count' matters: a big risk ratio on a handful of customers is just noise.
for c in categorical:
    print(c)
    df_group = df_full_train.groupby(c).churn.agg(['mean', 'count'])
    df_group['diff'] = df_group['mean'] - global_churn    # group minus overall
    df_group['risk'] = df_group['mean'] / global_churn    # ratio to overall
    display(df_group)
    print()

## 3.6 Feature importance: mutual information

In [ ]:
from sklearn.metrics import mutual_info_score

# Mutual information: how much does knowing the feature reduce our uncertainty about churn?
#   0 = independent (useless feature), larger = more informative. It is symmetric: MI(a, b) == MI(b, a).
# It compares category labels, so it only suits categorical variables (numbers need binning first).
print(mutual_info_score(df_full_train.churn, df_full_train.contract))
print(mutual_info_score(df_full_train.contract, df_full_train.churn))   # same value: symmetric
print(mutual_info_score(df_full_train.gender, df_full_train.churn))     # near 0: gender says almost nothing

In [ ]:
def mutual_info_churn_score(series):
    # apply() hands each column over as a Series; score it against the churn target
    return mutual_info_score(series, df_full_train.churn)

# One MI score per categorical feature, ranked: the top ones are the most informative about churn.
mi = df_full_train[categorical].apply(mutual_info_churn_score)
mi.sort_values(ascending=False)

## 3.7 Feature importance: correlation

In [ ]:
# Correlation (Pearson) measures linear association between two NUMERIC variables, from -1 to +1.
# With a 0/1 target it is still meaningful: positive = churn goes up as the feature goes up.
# .abs() keeps only the strength, dropping the sign.
df_full_train[numerical].corrwith(df_full_train.churn).abs()

In [ ]:
# Same idea by hand: churn rate in bands of tenure. Short tenure -> high churn, long tenure -> low.
# (& means AND in pandas filters, with parentheses around each condition.)
print('tenure <= 2        ', df_full_train[df_full_train.tenure <= 2].churn.mean())
print('2 < tenure <= 12   ', df_full_train[(df_full_train.tenure > 2) & (df_full_train.tenure <= 12)].churn.mean())
print('tenure > 12        ', df_full_train[df_full_train.tenure > 12].churn.mean())
print()
print('monthly <= 20      ', df_full_train[df_full_train.monthlycharges <= 20].churn.mean())
print('20 < monthly <= 50 ', df_full_train[(df_full_train.monthlycharges > 20) & (df_full_train.monthlycharges <= 50)].churn.mean())
print('monthly > 50       ', df_full_train[df_full_train.monthlycharges > 50].churn.mean())

## 3.8 One-hot encoding

In [ ]:
from sklearn.feature_extraction import DictVectorizer

# A model needs numbers. DictVectorizer turns each text value into its own 0/1 column
# ('contract=month-to-month', 'contract=one_year', ...) and passes numeric features through unchanged.
# sparse=False returns an ordinary numpy array instead of a memory-saving sparse matrix.
dv = DictVectorizer(sparse=False)

# Input format: a list of dicts, one dict per customer {feature: value}.
train_dict = df_train[categorical + numerical].to_dict(orient='records')

# fit_transform = LEARN the column layout from the TRAINING data, then encode it.
X_train = dv.fit_transform(train_dict)

# Validation is only transform()-ed: same columns, same order, nothing learned from validation.
val_dict = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dict)

X_train.shape, X_val.shape

## 3.9 Logistic regression

In [ ]:
def sigmoid(z):
    # Squashes any number into (0, 1), so a score can be read as a probability.
    return 1 / (1 + np.exp(-z))

z = np.linspace(-7, 7, 51)    # 51 evenly spaced points from -7 to 7
plt.plot(z, sigmoid(z))       # S-shaped curve: 0 on the far left, 0.5 at z=0, 1 on the far right
plt.xlabel('score z'); plt.ylabel('sigmoid(z)');

In [ ]:
# Linear regression: weighted sum of the features plus a bias (w0). Output is any real number.
def linear_regression(xi, w0, w):
    result = w0
    for j in range(len(w)):
        result = result + xi[j] * w[j]
    return result

# Logistic regression: the SAME weighted sum, then squashed by the sigmoid -> a probability in (0, 1).
def logistic_regression(xi, w0, w):
    score = linear_regression(xi, w0, w)
    return sigmoid(score)

## 3.10 Training logistic regression with scikit-learn

In [ ]:
# NOTE: you will see a ConvergenceWarning. The numeric columns (tenure up to 72, monthlycharges up to ~120,
# totalcharges in the thousands) have a very different scale from the 0/1 one-hot columns, so lbfgs needs many
# iterations. The accuracy is fine; the warning is the hint that scaling helps (see the scaling lecture notebook).
from sklearn.linear_model import LogisticRegression

# lbfgs is the default solver in current sklearn. HW3 requires
#   LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model = LogisticRegression(solver='lbfgs', max_iter=1000)
model.fit(X_train, y_train)    # learns the bias (intercept_) and one weight per column of X_train

In [ ]:
print('bias w0:', model.intercept_[0])
# One weight per column, in the same order as dv.get_feature_names_out().
# Positive weight pushes the churn probability up, negative pushes it down.
model.coef_[0].round(3)

In [ ]:
# predict_proba returns two columns: [P(stays), P(churns)]. [:, 1] keeps the churn probability.
y_pred = model.predict_proba(X_val)[:, 1]

# Turn probabilities into a yes/no decision with a threshold (0.5 here).
churn_decision = (y_pred >= 0.5)

# Accuracy = share of customers where the decision equals the actual outcome.
(y_val == churn_decision).mean()

In [ ]:
# The same accuracy built step by step, to inspect individual predictions.
df_pred = pd.DataFrame()
df_pred['probability'] = y_pred
df_pred['prediction'] = churn_decision.astype(int)
df_pred['actual'] = y_val
df_pred['correct'] = df_pred.prediction == df_pred.actual
print(df_pred.correct.mean())
df_pred.head()

## 3.11 Model interpretation

In [ ]:
# Pair each feature name with its weight to read the model.
# zip() walks two lists in parallel, dict() turns the pairs into {name: weight}.
dict(zip(dv.get_feature_names_out(), model.coef_[0].round(3)))

In [ ]:
# A smaller model with 3 features, to see exactly how a prediction is built.
small = ['contract', 'tenure', 'monthlycharges']

dicts_train_small = df_train[small].to_dict(orient='records')
dicts_val_small = df_val[small].to_dict(orient='records')

dv_small = DictVectorizer(sparse=False)
dv_small.fit(dicts_train_small)
X_train_small = dv_small.transform(dicts_train_small)
X_val_small = dv_small.transform(dicts_val_small)

model_small = LogisticRegression(solver='lbfgs', max_iter=1000)
model_small.fit(X_train_small, y_train)

w0 = model_small.intercept_[0]
w = model_small.coef_[0]
dict(zip(dv_small.get_feature_names_out(), w.round(3)))

In [ ]:
# The lecture types the weights in by hand (-2.47 + -0.949 + 30*0.027 + ...). Here the same calculation
# is done on the first validation customer, with the weights of YOUR model:
#   score = w0 + sum(feature * weight)   then   probability = sigmoid(score)
x0 = X_val_small[0]
score = w0 + (x0 * w).sum()
print('score:', score, ' manual probability:', sigmoid(score))

# It must equal what scikit-learn returns: predict_proba does exactly this.
print('sklearn probability:', model_small.predict_proba(x0.reshape(1, -1))[0, 1])

## 3.12 Using the model

In [ ]:
# NOTE: you will see a ConvergenceWarning. The numeric columns (tenure up to 72, monthlycharges up to ~120,
# totalcharges in the thousands) have a very different scale from the 0/1 one-hot columns, so lbfgs needs many
# iterations. The accuracy is fine; the warning is the hint that scaling helps (see the scaling lecture notebook).
# Final training: use train + validation together (more data), and evaluate ONCE on the test set.
dicts_full_train = df_full_train[categorical + numerical].to_dict(orient='records')

dv = DictVectorizer(sparse=False)
X_full_train = dv.fit_transform(dicts_full_train)    # layout learned from full_train only
y_full_train = df_full_train.churn.values

model = LogisticRegression(solver='lbfgs', max_iter=1000)
model.fit(X_full_train, y_full_train)

dicts_test = df_test[categorical + numerical].to_dict(orient='records')
X_test = dv.transform(dicts_test)                    # transform only: nothing learned from test

y_pred = model.predict_proba(X_test)[:, 1]
churn_decision = (y_pred >= 0.5)
(churn_decision == y_test).mean()                    # test accuracy

In [ ]:
# Predicting for ONE customer, as a web service would: a single dict of features.
customer = dicts_test[-1]
customer

In [ ]:
# transform() expects a LIST of dicts, so wrap the customer in [...]; the result is a 1-row matrix.
X_one = dv.transform([customer])

# [0, 1]: row 0 (the only row), column 1 (probability of churn).
print('predicted churn probability:', model.predict_proba(X_one)[0, 1])
print('actual outcome:', y_test[-1])